<br>
<a href="https://www.nvidia.com/en-us/training/">
    <div style="width: 55%; background-color: white; margin-top: 50px;">
    <img src="https://dli-lms.s3.amazonaws.com/assets/general/nvidia-logo.png"
         width="400"
         height="186"
         style="margin: 0px -25px -5px; width: 300px"/></div>
</a>
<h1 style="line-height: 1.4;"><font color="#76b900"><b>Building LLM Applications With Prompt Engineering</b></font></h1>
<h2><b>Reasoning Models</b></h2>
<br>


<!-- REASONING_MODEL_POLICY_NEMOTRON_20260505 -->
This notebook intentionally uses two different Nemotron models:

- `nvidia/nemotron-nano-12b-v2-vl` is the direct-response baseline. It is the right default for ordinary prompt iteration, structured output, routing, and tool-use examples.
- `nvidia/nemotron-3-nano-omni-30b-a3b-reasoning` is reserved for tasks where the extra deliberation is the point. It can process text, image, audio, and video inputs, but here you use it to study reasoning behavior through the same chat interface.

Do not treat the reasoning model as the default model for the whole course. The lesson is to choose it only when the task justifies the additional latency, output length, and parsing care.



In the previous notebook, we learned about Chain-of-Thought (CoT) prompting, a technique where we *prompt* an LLM to think step by step. But what if the model could do this automatically, without needing to be prompted?

That's exactly what **reasoning models** are designed to do. In this notebook, we'll explore these specialized models, understand their trade-offs, and learn how to work with them in practice.


```mermaid
flowchart TB
    subgraph N["Non-reasoning call"]
      N1["Prompt"]:::box --> N2["Model"]:::box --> N3["Visible answer text"]:::answer
    end

    subgraph R["Reasoning call"]
      R1["Prompt"]:::box --> R2["Reasoning model"]:::gpu
      R2 --> R3["Visible answer text"]:::answer
      R2 -.-> R4["Reasoning metadata<br>effort / tokens / summaries"]:::hidden
    end

    classDef box fill:#161b22,stroke:#30363d,stroke-width:1.5px,color:#e6edf3
    classDef gpu fill:#211b12,stroke:#f2cc60,stroke-width:2px,color:#f2cc60
    classDef answer fill:#0d2b40,stroke:#58a6ff,stroke-width:2px,color:#58a6ff
    classDef hidden fill:#1f1a2e,stroke:#8b5cf6,stroke-width:2px,color:#c4b5fd,stroke-dasharray:5 5
```

---


## Objectives


By the time you complete this notebook you will:

- Understand what reasoning models are and how they differ from standard LLMs
- Recognize the trade-offs involved in using reasoning models
- Learn how to handle the "thinking" tokens that reasoning models produce
- Be able to integrate reasoning models into LangChain chains


---


## Imports


In [ ]:
import os
import re
import time

from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnableLambda
from langchain_core.messages import AIMessage

---


## Create Model Instances


In this notebook, we'll be comparing a standard LLM with a reasoning model. Let's create both.

For our reasoning model, we'll use [**`nvidia/nemotron-3-nano-omni-30b-a3b-reasoning`**](https://build.nvidia.com/nvidia/nemotron-3-nano-omni-30b-a3b-reasoning). It is NVIDIA's Nemotron 3 Nano Omni reasoning model: a 30B-total, 3B-active omnimodal model intended for tasks that benefit from deliberate analysis across rich inputs.


In [ ]:
base_url = os.getenv("NVIDIA_BASE_URL")

# Our standard model (same as previous notebooks)
standard_llm = ChatNVIDIA(base_url=base_url, model='nvidia/nemotron-nano-12b-v2-vl', temperature=0)

# A reasoning model
reasoning_llm = ChatNVIDIA(
    base_url=base_url,
    model='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning',
    temperature=0.5,             # See model card for task-specific temperature/sampling parameter recommendations.
    max_completion_tokens=2048,  # Reasoning models need enough room for analysis plus the final answer.
)

---


## Streaming Printing Helper

In this notebook we will use the following helper function to print streaming responses from the LLM.

In [ ]:
def sprint(stream):
    for chunk in stream:
        if chunk.additional_kwargs.get('reasoning'):            ## If there is reasoning, we will display it.
            print(chunk.additional_kwargs.get('reasoning', ""), end='', flush=True)
        if chunk.content:
            print("\033[1m" + chunk.content + "\033[0m", end='', flush=True) ## Otherwise, we will print output in bold.

---

## Reasoning Models

In a previous notebook, you learned that standard LLMs tend to "jump to conclusions" because they're designed to predict the most likely next token. We showed how Chain-of-Thought prompting can help by explicitly asking the model to think step by step.

**Reasoning models** take this a step further. They are trained (or fine-tuned) to *automatically* engage in extended internal reasoning before producing their final answer. You don't need to prompt them to think step by step: they do it naturally.

Let's see this in action with a puzzle requiring some multi-step logical thinking. We'll present both models with the same problem and compare their responses.

In [ ]:
problem = """You are a world-class logician. Read the following short story carefully:
Alice, Bob, and Charlie are the only three people in a house. At midnight, exactly one of them is in the kitchen, exactly one is in the bedroom, and exactly one is in the bathroom.
The light in the kitchen is on.
Alice hates bright lights and never enters a room that has its light on.
Bob is afraid of the dark and never enters a room that has its light off.
Charlie always tells the truth.
Charlie says: “Bob is in the bathroom.”
Where is each person, and is the bedroom light on or off?
Answer with exactly three sentences: one stating where Alice is, one stating where Bob is, and one stating where Charlie is, plus a fourth sentence about the bedroom light.
Show every step of your reasoning first. Do not guess."""

If you're interested try to work the problem out for yourself. When you're ready view the solution by viewing the cell immediately below.

<details><summary><b>Click to show solution</b></summary>
<blockquote><pre>Alice is in the bedroom.  
Bob is in the bathroom.  
Charlie is in the kitchen.  
The bedroom light is off.
</pre></blockquote></details>

First, let's see how our non-reasoning model performson this tricky logic puzzle.

In [ ]:
## Diagnostic Check: What's actually coming out of the stream.
# stream = standard_llm.stream(problem)
# print(next(stream))
# print(next(stream))
sprint(standard_llm.stream(problem))

And now let's give the problem to a reasoning model.

In [ ]:
## Diagnostic Check: What's actually coming out of the stream.
# stream = reasoning_llm.stream(problem)
# print(next(stream))
# print(next(stream))

sprint(reasoning_llm.stream(problem))

Notice the difference? The standard model gives a quick, direct answer. The reasoning model, on the other hand, produces a much longer response that includes its step-by-step thinking process.

**Why is reasoning coming as a new argument:** If you looked behind the server, the reasoning model wraps its internal deliberation in `<think></think>` tags or similar. This is the model "thinking out loud" before giving its final answer. The API parses this out and delivers it through a different pathway, hence the additional argument which gets returned per chunk.


---


## The Trade-Offs


If reasoning models are better at complex problems, why not use them for everything? There are important trade-offs to consider:

1. **Latency**: Reasoning models take longer to respond because they generate many more tokens.
2. **Cost**: More tokens means higher costs (in API-based scenarios) or more compute (in self-hosted scenarios).
3. **Overkill for simple tasks**: For straightforward questions (like the example above), the extra reasoning adds no value.

Let's measure the latency difference on a simple task.

In [ ]:
simple_task = "Summarize the benefits of exercise in one sentence."

# Time the standard model
start = time.time()
standard_response = standard_llm.invoke(simple_task)
standard_time = time.time() - start

# Time the reasoning model
start = time.time()
reasoning_response = reasoning_llm.invoke(simple_task)
reasoning_time = time.time() - start

standard_text = standard_response.content or ""
reasoning_text = reasoning_response.content or ""
reasoning_trace = reasoning_response.additional_kwargs.get("reasoning_content", "")

print(f"Standard model:  {standard_time:.1f} seconds, {len(standard_text)} answer characters")
print(f"Reasoning model: {reasoning_time:.1f} seconds, {len(reasoning_text)} answer characters")
print(f"Reasoning trace: {len(reasoning_trace)} hidden/side-channel characters")
print(f"\nThe reasoning model was {reasoning_time/standard_time:.1f}x slower for this simple task.")


As you can see, the reasoning model takes significantly longer, even for a task that doesn't require complex reasoning. This is an important consideration when designing your applications: **use reasoning models when the task genuinely benefits from extended deliberation**.


---


## Handling Thinking Tokens


As we saw above, our NIM-hosted model separates thinking into `additional_kwargs['reasoning_content']`, keeping `.content` clean. However, not all reasoning models or endpoints work this way. There are two common paradigms you'll encounter:

1. **Structured API fields** — The serving infrastructure parses thinking server-side and places it in a separate response field (like `reasoning_content`). This is what our NIM endpoint does.
2. **Inline `<think>` tags** — Some models or self-hosted endpoints leave `<think>...</think>` tags directly in the response text.

To build robust applications, it's good practice to handle both cases. Let's create a helper function that does this.

In [ ]:
test_message = "<think>Let me think about his for a second....</think>Thinking is great!"
re.sub(r'<think>.*?</think>', '', test_message, flags=re.DOTALL)

Let's create a simple helper function using the same method to work on our model respones.

In [ ]:
def strip_thinking(message):
    """Return the final answer while dropping model-specific reasoning traces.

    Some reasoning models emit <think>...</think> in visible content.
    Nemotron 3 Nano Omni exposes reasoning in additional_kwargs instead, while
    message.content carries the final answer.
    """
    content = message.content or ""
    cleaned = re.sub(r'<think>.*?</think>', '', content, flags=re.DOTALL)
    if '<think>' in cleaned:
        cleaned = re.sub(r'<think>.*', '', cleaned, flags=re.DOTALL)
    return AIMessage(content=cleaned.strip(), additional_kwargs={})


Let's test this helper with a fresh response from our reasoning model. Since our NIM endpoint separates thinking into `additional_kwargs`, the `.content` field is already clean. The `strip_thinking` function handles this gracefully, and will also work if you switch to a model that uses inline `<think>` tags.

In [ ]:
response = reasoning_llm.invoke("What is the capital of France?")

In [ ]:
print(response.content)
print("\nReasoning trace preview:")
print(response.additional_kwargs.get("reasoning_content", "")[:500])


In [ ]:
stripped_response = strip_thinking(response)

In [ ]:
print(stripped_response.content)

In [ ]:
visible_content = response.content or ""
reasoning_trace = response.additional_kwargs.get("reasoning_content", "")
print(f"Visible answer before stripping: {len(visible_content)} characters")
print(f"Reasoning trace side channel: {len(reasoning_trace)} characters")
print(f"After stripping: {len(stripped_response.content)} characters")


**Important Note:** The format of reasoning traces is not standardized across models. Some models use visible `<think>` tags; Nemotron 3 Nano Omni exposes reasoning in response metadata such as `reasoning_content` while keeping the final answer in `content`. Always check the model documentation and inspect one response before you wire a reasoning model into a parser or tool call.

That said, the principle remains the same: you'll need to handle this content appropriately for your use case.


---


## Integrating into Chains


Now, how do we use our `strip_thinking` function in a LangChain chain? If you recall from earlier notebooks, we can use `RunnableLambda` to wrap any Python function and make it part of a chain.

Let's build a chain that uses the reasoning model and cleanly strips out the thinking process.


In [ ]:
def add_think_scope(message):
    message.content = "<think>Let me think first... I got it!</think>" + message.content
    return message

clean_reasoning_chain = reasoning_llm | add_think_scope | RunnableLambda(strip_thinking) | StrOutputParser()

In [ ]:
result = clean_reasoning_chain.invoke("In one sentence, what's the best thing anyone of any age can do?")
print(result)

The chain works as follows:
1. `reasoning_llm` generates a response with a final answer plus model-specific reasoning metadata.
2. `RunnableLambda(strip_thinking)` keeps the final answer and drops visible or side-channel reasoning traces.
3. `StrOutputParser()` extracts the clean string content.

The user or downstream runnables in the chain see only the polished final answer, while the model still benefits from its internal reasoning process.


---


## Why Stripping Matters: Structured Output


You might wonder: "Why bother stripping the thinking tokens? Can't I just ignore them?"

For some use cases, you can. But for others, particularly structured output parsing (which we'll be looking at in greater depth later in the workshop), the thinking tokens will break your pipeline. Let's see this in action.

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Respond with JSON only, no additional text."),
    ("human", "Extract name and age as JSON: {text}")
])

parser = JsonOutputParser() # Will parse key entities in the final output into a JSON object

First, let's try **without** stripping the thinking tokens.


In [ ]:
chain_without_strip = prompt | add_think_scope | reasoning_llm | parser

try:
    result = chain_without_strip.invoke({"text": "John Smith is 35 years old."})
    print(f"Result: {result}")
except Exception as e:
    print(f"Error: {type(e).__name__}")
    print("The <think> block broke the JSON parser!")

Now let's try **with** our `strip_thinking` function in the chain.


In [ ]:
chain_with_strip = (
    prompt | reasoning_llm 
    | add_think_scope 
    | strip_thinking
    | parser
)

result = chain_with_strip.invoke({"text": "John Smith is 35 years old."})
print(f"Result: {result}")

By stripping the thinking tokens before the JSON parser, everything works smoothly. This is a practical example of why understanding how to handle reasoning model output is essential for building robust applications.


---


## When to Keep the Thinking


We've focused on stripping thinking tokens, but sometimes you actually *want* to show them to users. For example:

- **Educational applications**: Showing students how the model reasoned through a problem
- **Transparency**: Letting users see the model's thought process builds trust
- **Debugging**: Understanding why a model produced a particular answer

In a chatbot interface, you might format the thinking in a collapsible section or display it with visual distinction. The choice depends entirely on your application's goals.


---


## Exercise: Build a Reasoning Chain


For this exercise, you'll build a chain that uses the reasoning model to perform a math problem beyond the scope of the non-reasoning Nemotron Nano 12B-V2-VL model, strips the thinking tokens, and returns just the clean answer.

Here's the problem:

In [ ]:
challenging_expression = "286899 * 3902789"

Let's see how our `standard_llm` does with this problem.

In [ ]:
# print(standard_llm.invoke(f"What is {challenging_expression}?").content)
sprint(standard_llm.stream(f"What is {challenging_expression}?"))
print(f"ACTUAL ANSWER: {challenging_expression} = {eval(challenging_expression):,}")

Your task:
1. Create a chain using `reasoning_llm`
2. Include `strip_thinking` in the chain to remove thinking tokens
3. Use `StrOutputParser` to get clean string output
4. Invoke the chain with the provided problem

Feel free to check out the *Solution* below if you get stuck.

### Your Work Here


In [ ]:
# Your code here

### Solution


In [ ]:
# A reasoning model
reasoning_llm = ChatNVIDIA(
    base_url=base_url,
    model='nvidia/nemotron-3-nano-omni-30b-a3b-reasoning',
    temperature=0.5,
    max_completion_tokens=8192,
)

sprint(reasoning_llm.stream(f"What is {challenging_expression}?"))
print(f"\nACTUAL ANSWER: {challenging_expression} = {eval(challenging_expression):,}")

---


## Summary


In this notebook, you learned:

- **Reasoning models** are LLMs trained to automatically engage in step-by-step deliberation before answering
- Their thinking may appear as inline `<think>...</think>` tags or in a separate response field like `reasoning_content`, depending on the model and serving infrastructure
- There are **trade-offs**: higher latency, more tokens, but better reasoning on complex tasks
- You can use `RunnableLambda` to strip thinking tokens when needed (especially for structured output)
- Sometimes you *want* to keep the thinking visible for transparency or education

Reasoning models represent an exciting evolution in LLM capabilities. As you build applications, consider when the added reasoning power is worth the trade-offs, and when a standard model with good prompting might be sufficient.

---

<!-- NEXT_STEP_CARD -->

<div style="border-left: 6px solid #76B900; background: #f7fdf2; padding: 14px 18px; border-radius: 10px; margin: 20px 0;">
<p style="margin: 0 0 6px; color: #315c00; font-weight: 700; letter-spacing: .04em; text-transform: uppercase;">Continue the unified course path</p>
<p style="margin: 0 0 8px; color: black;"><strong>Next step:</strong> Open <code>3-Prompting-With-Messages/36-Chatbots.ipynb</code> next: <strong>Chatbots</strong>.</p>
<p style="margin: 0; color: black;"><strong>Before moving on:</strong> Keep one judgment about when hidden thinking is worth the latency, because the next notebook folds those message and reasoning choices into a longer conversational assistant.</p>
</div>